In [1]:
import geopandas as gpd
import ee
import geemap
# ee.Authenticate()
# ee.Initialize()
import ipyleaflet
import shapely
import json
import rasterio
import matplotlib.pyplot as plt

In [2]:
#create empty map
Map = geemap.Map()

gdf=gpd.read_file('../../data/merged_jaipur_all.shp')

jaipur_json=json.loads(shapely.to_geojson(gdf['geometry'].values[0]))

roi= ee.Geometry.MultiPolygon(jaipur_json['coordinates'])
roi

# center map on selected country
Map.centerObject(roi, 7)

Map

Map(center=[27.262634207359138, 75.87933112276701], controls=(WidgetControl(options=['position', 'transparent_…

In [3]:
landcover= ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1").filterDate("2015-01-1","2023-05-19").filterBounds(roi).select('label')

In [4]:

crop_class = 4
# Create a binary layer where 1 represents crop and 0 represents all other landcover types
landcover_crop = landcover.map(lambda image: image.eq(crop_class))

# Define the visualization parameters for the crop layer
cropVis = { "min":0, "max":1, 'palette': ['black', '#E49635']}

# Add the crop layer to the map
Map.addLayer(landcover_crop.mosaic().clip(roi).selfMask(), cropVis, "Crop Landcover")

legend_dict = {
    'Crops': '#E49635',
}

Map.add_legend(legend_title="Land Cover of Jaipur", legend_dict=legend_dict)

In [5]:
crops= landcover_crop.mosaic().clip(roi).selfMask()
vectors= crops.reduceToVectors(eightConnected= False, geometryType= 'polygon', crs=crops.projection(), scale=1000)

In [6]:
drawing=vectors.draw(color= '800080', pointRadius= 5, strokeWidth= 3)
Map.addLayer(drawing)

In [19]:
vectors.size()

In [7]:
roi2=vectors.toList(5).get(0)

In [8]:
cropVis2 = { "min":0, "max":1, 'palette': ['black', '#34EBE1']}

# Add the crop layer to the map
Map.addLayer(landcover_crop.mosaic().clip(roi2).selfMask(), cropVis2, "Crop Landcover")
legend_dict2 = {
    'Crops': '#34EBE1',
}

Map.add_legend(legend_title="Land Cover of Jaipur", legend_dict=legend_dict2)

In [31]:
roi2

In [37]:
vectors.toList(5).get(0).getInfo()

{'type': 'Feature',
 'geometry': {'geodesic': False,
  'type': 'Polygon',
  'coordinates': [[[74.96441045977407, 27.030306899156404],
    [74.97339361261527, 27.030306899156404],
    [74.97339361261527, 27.039290051997597],
    [74.96441045977407, 27.039290051997597],
    [74.96441045977407, 27.030306899156404]]]},
 'id': '+8345+3009',
 'properties': {'count': 1, 'label': 1}}

In [50]:
allvectors=vectors.getInfo()

KeyboardInterrupt: 

In [40]:
len(allvectors['features'])

94

In [41]:
gdf=gpd.GeoDataFrame.from_features(allvectors['features']) #this is where it could be made higher resolution

In [55]:
gdf.explore()

In [54]:
gdf.crs='EPSG:4326'

In [56]:
gdf.to_file('jaipur_crops_rough.shp')